In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
%matplotlib inline

CSVS = ['circuits.csv','constructor_results.csv','constructor_standings.csv','constructors.csv',
        'driver_standings.csv','drivers.csv','lap_times.csv','pit_stops.csv','qualifying.csv',
        'races.csv','results.csv','sprint_results.csv','status.csv']

data = {}
for csv in CSVS:
    key = csv.replace('.csv','')
    data[key] = pd.read_csv(csv)

#Setting convenient key names
circuits          = data['circuits']
constructor_res   = data['constructor_results']
constructor_stand = data['constructor_standings']
constructors      = data['constructors']
driver_stand      = data['driver_standings']
drivers           = data['drivers']
lap_times         = data['lap_times']
pit_stops         = data['pit_stops']
qualifying        = data['qualifying']
races             = data['races']
results           = data['results']
sprint_results    = data['sprint_results']
status            = data['status']


## Testing Different Analysis Ideas

### Average Lap Time Per Year

In [ ]:
# --- Average Fastest Lap Time Per Year ---

# Grab only the columns we need from race results
fastest_laps = results[['raceId', 'fastestLapTime']].copy()

# Keep only rows that have a real lap time (contains a colon like "1:25.892")
has_valid_time = fastest_laps['fastestLapTime'].apply(lambda x: ':' in str(x))
fastest_laps = fastest_laps[has_valid_time]


def lap_time_to_seconds(time_str):
    """Convert a lap time string like '1:25.892' into total seconds (85.892)."""
    try:
        minutes_str, seconds_str = str(time_str).split(':')
        minutes = int(minutes_str)
        seconds = float(seconds_str)
        return minutes * 60 + seconds
    except (ValueError, AttributeError):
        return np.nan


# Convert every fastest-lap string into seconds
fastest_laps['fastest_lap_seconds'] = fastest_laps['fastestLapTime'].apply(lap_time_to_seconds)

# Drop rows where the conversion failed
fastest_laps = fastest_laps.dropna(subset=['fastest_lap_seconds'])

# Attach the year (and race name) so we can group by season
fastest_laps = fastest_laps.merge(races[['raceId', 'year', 'name']], on='raceId')

# Average fastest-lap time per year
avg_fastest_lap_by_year = fastest_laps.groupby('year')['fastest_lap_seconds'].mean().dropna()

# Plot
plt.figure(figsize=(14, 5))
avg_fastest_lap_by_year.plot()
plt.title('Average Fastest Lap Time per Season (seconds)')
plt.ylabel('Time (seconds)')
plt.xlabel('Year')
plt.show()